# Basic Queries

Uses cleaned data from **../../Part 2/cleaned_data** and the SQLite database in **../database/ecommerce.db**.

In [1]:
"""
Week 8 Mini Project - E-Commerce Order Analytics System
Part 3: SQL Analysis - Batch 1 (Basic Queries)

  1. Total revenue per category
  2. Top 10 customers by total order value
  3. Month-wise order count for the last 12 months

Design note: every revenue query here joins order_items -> orders (not just
order_items -> products), even when the orders columns aren't in the SELECT.
That inner join is what naturally excludes the 8 orphaned order_item rows
from Part 1/2 (the ones whose order_id doesn't exist in orders) - if we
only joined to products, those orphan rows would still count.
"""

from pathlib import Path
import sqlite3
import pandas as pd

DB_PATH = Path("../database/ecommerce.db")
conn = sqlite3.connect(DB_PATH)

def run_query(conn, sql):
    return pd.read_sql_query(sql, conn)


REVENUE_EXPR = "oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100.0)"


def query_1_revenue_per_category(conn):
    sql = f"""
        SELECT
            p.category,
            ROUND(SUM({REVENUE_EXPR}), 2) AS total_revenue,
            COUNT(DISTINCT oi.order_id) AS orders_involved
        FROM order_items oi
        JOIN products p ON oi.product_id = p.product_id
        JOIN orders o   ON oi.order_id   = o.order_id
        GROUP BY p.category
        ORDER BY total_revenue DESC;
    """
    return run_query(conn, sql)


def query_2_top_10_customers(conn):
    sql = f"""
        SELECT
            CAST(o.customer_id AS INTEGER) AS customer_id,
            c.customer_name,
            ROUND(SUM({REVENUE_EXPR}), 2) AS total_order_value
        FROM order_items oi
        JOIN orders o    ON oi.order_id    = o.order_id
        JOIN customers c ON o.customer_id  = c.customer_id
        WHERE o.customer_id IS NOT NULL
        GROUP BY o.customer_id, c.customer_name
        ORDER BY total_order_value DESC
        LIMIT 10;
    """
    return run_query(conn, sql)


def query_3_monthly_order_count_last_12_months(conn):
    # anchor "last 12 months" to the latest order_date actually in the data
    # rather than hardcoding a date, so this keeps working no matter when
    # the dataset is regenerated
    sql = """
        WITH cutoff AS (
            SELECT date(MAX(order_date), '-12 months') AS cutoff_date
            FROM orders
        )
        SELECT
            strftime('%Y-%m', order_date) AS year_month,
            COUNT(*) AS order_count
        FROM orders, cutoff
        WHERE order_date >= cutoff.cutoff_date
        GROUP BY year_month
        ORDER BY year_month;
    """
    return run_query(conn, sql)


def main():
    conn = sqlite3.connect(DB_PATH)
    
    print("\n=== Query 1: Total Revenue per Category ===")
    print(query_1_revenue_per_category(conn).to_string(index=False))

    print("\n=== Query 2: Top 10 Customers by Total Order Value ===")
    print(query_2_top_10_customers(conn).to_string(index=False))

    print("\n=== Query 3: Month-wise Order Count (Last 12 Months) ===")
    print(query_3_monthly_order_count_last_12_months(conn).to_string(index=False))

    conn.close()


if __name__ == "__main__":
    main()



=== Query 1: Total Revenue per Category ===
   category  total_revenue  orders_involved
Electronics    60699202.68              627
       Home    11849597.12              724
     Sports     9890898.33              667
   Clothing     5767651.01              679
     Beauty     4230646.46              769
      Books     1342313.53              702

=== Query 2: Top 10 Customers by Total Order Value ===
 customer_id   customer_name  total_order_value
         387   Angela Willis         1007976.38
          79   Marie Gilbert          992841.92
         201      Steve Paul          664474.92
         529 Krystal Brennan          664414.68
         158  Jeremy Coleman          614225.31
         332  Justin Delgado          612828.43
         524  Raymond Tucker          556640.79
         222   Karen Ballard          550982.63
         182  Brittany White          549294.24
          73 Michael Elliott          547619.44

=== Query 3: Month-wise Order Count (Last 12 Months) ===
year_